##### Notebook to scrape reviews for a certain game and store the results in a csv file

In [1]:
appid = 3606480  #BO7  
game_name = "BlackOps7"
filter = "recent" # recent, updated or all (i.e. sorted by helpfulness)
                  # NOTE: day_range is only respected by "recent" and "updated", not "all"
day_range = 249 # reviews up to X days in the past
page_limit = 99999 # each page is 100 reviews
min_word_length = 30 # skip reviews with less than X words
csv_filename = f'reviews_{game_name}.csv'



In [2]:
from datetime import datetime, date

# Ask user for a start date and compute day_range
start_date_str = input("Scrape reviews from (YYYY-MM-DD): ").strip()
start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
day_range = (date.today() - start_date).days

print(f"Scraping reviews from {start_date} ({day_range} days ago) to today")



Scraping reviews from 2025-10-10 (273 days ago) to today


### Configuration Notes

**`filter` options:**
| Value | Sort order | `day_range` respected? | Notes |
|-------|-----------|----------------------|-------|
| `recent` | Newest first | ✅ Yes | Best for bulk scraping — cursor pagination works correctly across all pages |
| `updated` | Recently edited first | ✅ Yes | Similar to `recent` but surfaces re-edited reviews |
| `all` | Most helpful first | ❌ No | Steam only exposes  105 reviews before cursor-looping; not suitable for large scrapes |

**Recommendation:** Use `filter = "recent"` with a `day_range` to get a large, clean, date-bounded dataset.


In [9]:
# remove csv file to start from scratch
import os
if os.path.exists(csv_filename):
    os.remove(csv_filename)
    print(f"Removed existing {csv_filename}")


Removed existing reviews_BlackOps7.csv


In [10]:
import csv
import os
from datetime import datetime

def append_reviews_to_csv(data, csv_filename, min_word_length=30):
    """
    Appends review data to a CSV file.
    Creates the file with headers if it doesn't exist.
    
    Args:
        data: The response data from Steam API containing reviews
        csv_filename: Name of the CSV file to append to
        min_word_length: Minimum number of words required for a review
    """
    # Check if file exists to determine if we need to write headers
    file_exists = os.path.isfile(csv_filename)
    
    # Open file in append mode
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['review', 'author_vote', 'other_votes', 'weighted_vote_score', 'review_date']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # Write header if file is new
        if not file_exists:
            writer.writeheader()
        
        # Process each review in the page
        if 'reviews' in data:
            for review in data['reviews']:
                # Extract and clean the review text
                review_text = review['review']

                # text too short ?
                if len(review_text.split()) < min_word_length:
                    continue
                
                # Remove problematic characters that break CSV structure
                review_text = review_text.replace('\n', ' ')  # Replace newlines with spaces
                review_text = review_text.replace('\r', ' ')  # Replace carriage returns with spaces
                review_text = review_text.replace('\t', ' ')  # Replace tabs with spaces
                review_text = review_text.replace('"', '')    # Remove quotes
                
                # Remove unicode characters
                review_text = review_text.encode('ascii', 'ignore').decode('ascii')
                
                # Clean up multiple spaces
                review_text = ' '.join(review_text.split())
                
                # Convert author vote to 1 or 0
                author_vote = 1 if review['voted_up'] else 0
                
                # Convert Unix timestamp to readable date
                review_date = datetime.fromtimestamp(review['timestamp_created']).strftime('%Y-%m-%d')
                
                # Write row to CSV
                writer.writerow({
                    'review': review_text,
                    'author_vote': author_vote,
                    'other_votes': review['votes_up'],
                    'weighted_vote_score': review['weighted_vote_score'],
                    'review_date': review_date
                })


In [11]:
import requests, sys, time
from urllib.parse import unquote

cursor = '*'  # initial cursor
base_url = f'https://store.steampowered.com/appreviews/{appid}'

curr_page = 0
seen_cursors = set()

while True:
    params = { # https://partner.steamgames.com/doc/store/getreviews
        'json' : 1,
        'filter' : filter, # sort by: recent, updated, do not use all (i.e. by helpfulness) as it will only return the 100 "best" reviews
        'language' : 'english', # https://partner.steamgames.com/doc/store/localization
        'day_range' : day_range, # shows reviews from last X days, recent and updated only!
        'review_type' : 'all', # all, positive, negative
        'purchase_type' : 'steam', # all, non_steam_purchase, steam
        'num_per_page' : 100,
        'cursor': cursor,
    }
    response = requests.get(base_url, params=params)
    data = response.json()

    reviews = data.get('reviews', [])
    if not reviews:
        print("\nNo more reviews - all pages scraped.", file=sys.stderr)
        break

    append_reviews_to_csv(data, csv_filename, min_word_length)

    new_cursor = unquote(data['cursor'])

    if new_cursor == cursor:
        print("\nCursor unchanged - all pages scraped.", file=sys.stderr)
        break

    if new_cursor in seen_cursors:
        print("\nCursor loop detected - all unique pages scraped.", file=sys.stderr)
        break

    seen_cursors.add(cursor)
    cursor = new_cursor

    print(curr_page, end=" ", file=sys.stderr)
    curr_page += 1

    if curr_page >= page_limit:
        print(f"\nPage limit ({page_limit}) reached.", file=sys.stderr)
        break

    time.sleep(0.25)  # short pause to avoid rate limiting

print(f"\nReviews saved to {csv_filename}")


0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 


Reviews saved to reviews_BlackOps7.csv



No more reviews - all pages scraped.


In [12]:
import pandas as pd

csv_file = "reviews_BlackOps7.csv"
df = pd.read_csv(csv_file)

if "review" not in df.columns:
    raise KeyError("The CSV does not contain a 'review' column.")

word_counts = df["review"].fillna("").astype(str).str.split().str.len()
total_words = int(word_counts.sum())

print(f"Rows loaded: {len(df)}")
print(f"Total words in review column: {total_words}")
print(f"Average words per review: {word_counts.mean():.2f}")

Rows loaded: 1110
Total words in review column: 152059
Average words per review: 136.99
